# Chess Live Scanner — entrenamiento del clasificador de casillas (Kaggle / Colab)

Reproduce `python -m chessscan.pipeline` con GPU. Activar GPU (Kaggle: *Settings → Accelerator → GPU T4*; Colab: *Entorno de ejecución → Cambiar tipo → T4*) e Internet.

1. Subí la carpeta `ml/` del repo (como dataset de Kaggle o zip a Colab) **o** cloná el repo en la celda siguiente.
2. Ejecutá todo. Al final se descarga `run.zip` con el modelo `.tflite`, `metrics.json` y `labels.txt`.
3. Copiá el contenido a `ml/runs/<nombre>/` en la PC y seguí con `ml/README.md` (sección «Exportar»).

In [ ]:
# Opción A: clonar el repo (completar la URL si el repo está en GitHub)
REPO_URL = ""  # p. ej. "https://github.com/<usuario>/<repo>.git"
import os, subprocess
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
    ML = "repo/ml"
else:
    # Opción B: carpeta ml/ subida (Kaggle: /kaggle/input/<dataset>/ml ; Colab: descomprimir ml.zip en /content/ml)
    ML = next(p for p in ["/kaggle/input/chessscan-ml/ml", "/content/ml", "ml"] if os.path.isdir(p))
    if ML.startswith("/kaggle/input"):
        subprocess.run(["cp", "-r", ML, "ml"], check=True); ML = "ml"
os.chdir(ML); print(os.getcwd())

In [ ]:
!pip install -q "tensorflow[and-cuda]==2.19.1" numpy==2.1.3 opencv-python-headless==4.12.0.88 chess==1.11.2 tqdm==4.67.1
import tensorflow as tf; print(tf.__version__, tf.config.list_physical_devices("GPU"))

In [ ]:
# Dataset sintético de chesscog (OSF xf3ka, CC-BY 4.0), ~7.4 GB
!mkdir -p data/raw && cd data/raw &&   curl -sL -o val.zip https://osf.io/download/ydtcs/ &&   curl -sL -o test.zip https://osf.io/download/7p69s/ &&   curl -sL -o train.zip https://osf.io/download/z5vgm/ &&   curl -sL -o train.z01 https://osf.io/download/np56u/ && ls -l
# train está partido en dos volúmenes: unirlos y extraer
!cd data/raw && zip -s 0 train.zip --out train_full.zip && rm train.zip train.z01 &&   mkdir -p extracted && unzip -q val.zip -d extracted && unzip -q test.zip -d extracted && unzip -q train_full.zip -d extracted && rm *.zip && ls extracted

In [ ]:
!python -m chessscan.prepare_chesscog --raw data/raw/extracted --out data/prepared && rm -rf data/raw

In [ ]:
# Mismos hiperparámetros que el modelo entregado (ver ml/RESULTS.md). Semilla fija.
!python -m chessscan.pipeline --run runs/colab --data data/prepared --epochs 12 --steps 1500 --batch 128 --seed 1234 --workers 4

In [ ]:
import json; m = json.load(open("runs/colab/metrics.json"))
for k, v in m.items():
    if isinstance(v, dict) and "test" in v: print(k, v.get("bytes"), round(v["test"]["square_acc"], 5))
!cd runs && zip -qr ../run.zip colab
try:
    from google.colab import files; files.download("run.zip")
except ImportError:
    print("Kaggle: run.zip queda en /kaggle/working/" + "ml/run.zip (pestaña Output)")